In [1]:
import os
import pandas as pd

import matplotlib.pyplot as plt
import statistics
import numpy as np
import re

# Pre-processing

In [4]:
tpath =  "D:\\ACC Lab Dropbox\\ACC Lab\\Nicole Lee\\Worm tracking\ML\\"
filename = "20230309 KCRGS"
genotype = "KCR1-GS"
Path = tpath + filename + "\\"

In [5]:
rollingAverageWindow = 30
  
trainingList = [f for f in os.listdir(Path) if '.csv' in f]
dftest = pd.DataFrame()
for f in trainingList:
    traces = pd.read_csv(os.path.join(Path, f))
    title = f.split("downsampled")[0]
    title = title.replace("half", "0.5x")
    title = title.replace("WT_Green", "WT-Green")
    title = title.replace("WT_Blue", "WT-Blue")
    traces.columns = [title+ '_' + traces.iloc[0, i]+ '_' +  traces.iloc[1, i] for i in range(len(traces.columns))]
    traces = traces.drop([0, 1], axis = 0).reset_index().drop('index', axis = 1).astype('float').round(2)
    traces = traces.rename(columns = {traces.columns[0]: "frame"})
    traces = traces[traces.columns.drop(list(traces.filter(regex='likelihood.*')))]
   #traces = traces.rolling(rollingAverageWindow).mean()
    dftest = pd.concat([dftest, traces], axis = 1)

dftest = dftest.loc[:,~dftest.columns.duplicated()].reset_index(drop=True)
dftestt=pd.DataFrame()
dftestt = pd.concat([dftest.iloc[:,0], dftest.iloc[:,1:]*3], axis =1)  #pixel conversion is 1pix = 3uM

savefolder = tpath+ "Processed//" + title.split("_")[0] + '.csv'
dftestt.to_csv(savefolder)

# print(savefolder)

# formulas

In [6]:
def allmean(dfexpt):
    column_list = [dfexpt.iloc[:,k:k+8] for k in range(1, len(dfexpt.columns), 8)]
    dfss = pd.DataFrame()
    for i in range(0,len(column_list)):
        aa = column_list[i].columns.str.split("_").to_list()
        conc = aa[0][1]
        geno = aa[0][0]
        dfss[geno + ' ' + conc + '_' + 'meanX_'+ str(i+1)] = column_list[i].filter(regex = '_x').mean(axis = 1)
        dfss[geno + ' ' + conc + '_' +'meanY_'+ str(i+1)] = column_list[i].filter(regex = '_y').mean(axis = 1)
        dfss[geno + ' ' + conc + '_' +'meanSpeed_'+ str(i+1)] = (((np.sqrt(np.square(dfss[geno + ' ' + conc + '_' + 'meanX_'+ str(i+1)].diff())+np.square(dfss[geno + ' ' + conc + '_' +'meanY_'+ str(i+1)].diff()))))-0.15)
        dfss.loc[(dfss[geno + ' ' + conc + '_' +'meanSpeed_'+ str(i+1)]>0.8), [geno + ' ' + conc + '_' +'meanSpeed_'+ str(i+1)]] = np.nan
        

    tot = pd.DataFrame()
    tot['frame'] = dfexpt['frame']
    tot['second'] = dfexpt['frame']/29.2  #fps of camera
    tott = pd.DataFrame()
    speeds = dfss.filter(regex="meanSpeed.*").rolling(30,min_periods=1).mean() 
    tot['totalmeanspeed'] = speeds.mean(axis=1)
    tot['SEM'] = speeds.sem(axis=1)*1.96 
    tott = pd.concat([tot, dfss.filter(regex="meanSpeed.*")], axis = 1)
    tott = tott.iloc[:1723,:]

    return tott

In [7]:
def filterfour(dfs):
    p4 = dfs.filter(regex="bodypart4.*")
    p5 = dfs.filter(regex="bodypart5.*")
    p6 = dfs.filter(regex="bodypart6.*")
    p7 = dfs.filter(regex="bodypart7.*")

    df = pd.concat([dfs['frame'],p4,p5, p6,p7], axis =1)
    df = df.iloc[:1751,:]
    
    return df

# final processing

In [8]:
tpath1 =  "D:\\ACC Lab Dropbox\\ACC Lab\\Nicole Lee\\Worm tracking\ML\\Processed\\"
genotype = "KCR1GS"
wt_genotype = "WT-Blue"

In [9]:
dfexpt = pd.read_csv(tpath1 + genotype + ".csv")
dfexpt = filterfour(dfexpt)

dft0 = pd.DataFrame()
dft0 = pd.concat([dfexpt['frame'],dfexpt.filter(regex = "0x.*")], axis = 1)

dfwt = pd.read_csv(tpath1 + wt_genotype + ".csv")
dfwt = filterfour(dfwt)

In [10]:
conc = ["0x_", "0.5x_", "1x_", "2x_", "4x_"]
dfgenotype_0 = pd.concat([dfexpt['frame'],dfexpt.filter(regex = "0x_.*")], axis=1)    

cept = pd.DataFrame()
cewt = pd.DataFrame()
lstwt = []
for i in conc:

    if i == "0.5x_":
        label = "0.25mM"
    if i == "1x_":
        label = "0.5mM"
    if i == "2x_":
        label = "1mM"
    if i == "4x_":
        label = "2mM"
    
    crazyexpt = pd.DataFrame()
    ddd = pd.DataFrame()
    ddd = pd.concat([dfexpt['frame'], dfexpt.filter(regex = i + ".*")], axis=1)   
    crazyexpt = allmean(ddd.filter(regex="^((?!0x_).)*$"))
    
    totalcontrol = pd.concat([dfgenotype_0, dfwt.filter(regex = i + ".*")], axis=1)    
    crazywt= allmean(totalcontrol)
    
    dfo = crazywt.iloc[:291,:]  #normalizing speed from the first 10s
    value = dfo["totalmeanspeed"].mean()
    crazyexpt.iloc[:, 4:] -= value
    crazywt.iloc[:, 4:] -= value
    lstwt.append(value)

    cept = pd.concat([cept, crazyexpt.iloc[:,4:]], axis = 1)  
    cewt = pd.concat([cewt, crazywt.filter(regex = "WT-.*")], axis = 1)  

averagewtspeed = statistics.mean(lstwt)
df_0 = allmean(dfgenotype_0)
df_0.iloc[:, 4:] -= averagewtspeed

sync = pd.DataFrame()   
justval = pd.DataFrame()
justval = pd.concat([cept, cewt, df_0], axis = 1)
sync = pd.concat([crazyexpt.iloc[:,:2], justval.filter(regex="meanSpeed.*")], axis=1)

savefolder2 = tpath1+ "Reprocced\\" + genotype + '.csv'
sync.to_csv(savefolder2)

print(savefolder2)

D:\ACC Lab Dropbox\ACC Lab\Nicole Lee\Worm tracking\ML\Processed\Reprocced\KCR1GS.csv
